<a href="https://colab.research.google.com/github/dapenuela/Redes_Neuronales_V/blob/main/Redes_Neuronales_V_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Redes Neuronales V — Verificación facial con una red siamesa

Diego Armando Peñuela Cárdenas · Universidad de Cundinamarca

Abre este archivo en Colab. Ejecuta las celdas en orden con un entorno Python 3. La prueba matemática puede ejecutarse sin fotos. La evaluación facial requiere un dataset propio o autorizado. Evita guardar fotos, recortes, embeddings, pesos entrenados con datos privados y salidas identificables en GitHub. Publica el notebook con las salidas privadas borradas.

## 1. Organización del dataset privado

Coloca fotos JPG o PNG en `/content/dataset_privado/partición/id_anonimo/foto.jpg`. Usa el mismo identificador para la misma persona y asigna a cada persona a una sola partición. Cada foto debe mostrar un solo rostro frontal. Usa fotos independientes, idealmente de sesiones diferentes, y evita ráfagas casi idénticas. Los mínimos se aplican después de detectar los rostros.

| Partición | Identidades mínimas | Fotos válidas por identidad | Función |
|---|---:|---:|---|
| train | 2 | 2 | Ajustar pesos |
| cal | 2 | 4 | Seleccionar umbral |
| test | 2 | 4 | Medir one-shot y few-shot |
| unknown | 1 | 1 | Evaluar rechazo de desconocidos |

Se necesitan como mínimo 7 personas distintas y 21 fotos válidas. Es un mínimo técnico. Con muestras pequeñas, reporta los denominadores y evita generalizar. Guarda el consentimiento de forma privada y confirma su existencia antes de ejecutar la fase facial.


## 2. Prueba matemática de Triplet Loss

Se verifican los casos con y sin penalización mediante vectores sintéticos. Esta celda se ejecutó localmente y conserva su salida.

In [2]:
import numpy as np

a = np.array([1., 0.])
for positive, negative, expected in (
    (np.array([0., 1.]), np.array([1., 0.]), 2.2),
    (np.array([1., 0.]), np.array([0., 1.]), 0.0),
):
    d_ap = np.sum((a - positive)**2)
    d_an = np.sum((a - negative)**2)
    loss = max(d_ap - d_an + .2, 0.)
    np.testing.assert_allclose(loss, expected)
    print(f"d(A,P)²={d_ap:.1f} | d(A,N)²={d_an:.1f} | pérdida={loss:.1f}")


d(A,P)²=2.0 | d(A,N)²=0.0 | pérdida=2.2
d(A,P)²=0.0 | d(A,N)²=2.0 | pérdida=0.0


## 3. Codificador compartido y prueba de integración

Las tres ramas llaman a la misma instancia de Keras. La prueba de integración usa ruido, comprueba dimensiones y normalización, y ejecuta una actualización del optimizador en Colab.

In [3]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SIZE = 96
MARGIN = 0.2

def build_encoder():
    inputs = keras.Input((SIZE, SIZE, 3))
    x = inputs
    for filters in (32, 64, 128):
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, use_bias=False)(x)
    outputs = layers.UnitNormalization(axis=-1)(x)
    return keras.Model(inputs, outputs, name="codificador")

def triplet_loss(anchor, positive, negative):
    d_ap = tf.reduce_sum(tf.square(anchor - positive), axis=-1)
    d_an = tf.reduce_sum(tf.square(anchor - negative), axis=-1)
    return tf.reduce_mean(tf.maximum(d_ap - d_an + MARGIN, 0.0))

def train_step(encoder, optimizer, anchor, positive, negative):
    with tf.GradientTape() as tape:
        loss = triplet_loss(*(encoder(x, training=True)
                              for x in (anchor, positive, negative)))
    gradients = tape.gradient(loss, encoder.trainable_variables)
    optimizer.apply_gradients(zip(gradients, encoder.trainable_variables))
    return loss


In [4]:
keras.utils.set_random_seed(42)
a = tf.constant([[1., 0.]])
p = tf.constant([[0., 1.]])
n = tf.constant([[1., 0.]])
tf.debugging.assert_near(triplet_loss(a, p, n), 2.2)
tf.debugging.assert_near(triplet_loss(a, n, p), 0.)
encoder_demo = build_encoder()
rng_demo = np.random.default_rng(19)
noise_images = [rng_demo.uniform(0, 1, (4, SIZE, SIZE, 3)).astype(np.float32)
                for _ in range(3)]
before = int(encoder_demo.count_params())
demo_loss = train_step(encoder_demo, keras.optimizers.Adam(), *noise_images)
tf.debugging.assert_all_finite(demo_loss, "Pérdida inválida")
embeddings_demo = encoder_demo(noise_images[0])
tf.debugging.assert_equal(tf.shape(embeddings_demo), [4, 128])
tf.debugging.assert_near(tf.norm(embeddings_demo, axis=1), tf.ones(4))
print(f"Prueba de integración con ruido: {before:,} parámetros, pérdida {demo_loss.numpy():.4f}.")
print("Los tensores aleatorios no representan rostros ni miden reconocimiento.")


Prueba de integración con ruido: 2,452,544 parámetros, pérdida 0.2028.
Los tensores aleatorios no representan rostros ni miden reconocimiento.


## 4. Preprocesamiento y protocolo

Corrige orientación EXIF, detecta un único rostro, recorta, escala a 96 × 96 RGB y divide las intensidades por 255. Las etiquetas son identificadores anónimos. Los hashes impiden duplicados exactos, pero las fotos casi idénticas se deben revisar manualmente.

### Si Colab no encuentra `CascadeClassifier`

Crea una celda temporal y ejecuta estas líneas. Los paquetes de OpenCV comparten el módulo `cv2`. Reinstala solo la variante sin interfaz gráfica y conserva la versión de NumPy de Colab.

```python
%pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless opencv-contrib-python-headless
%pip install --no-deps --no-cache-dir "opencv-python-headless==4.12.0.88"
```

Reinicia la sesión de Colab y ejecuta de nuevo las celdas originales. Si el diagnóstico muestra un archivo `cv2.py` o una carpeta local `cv2` que creaste, renómbralo antes de reiniciar.

In [5]:

from collections import defaultdict
from pathlib import Path
import hashlib
import cv2
import numpy as np
from PIL import Image, ImageOps

if not hasattr(cv2, "CascadeClassifier") or not hasattr(cv2, "data"):
    raise RuntimeError(
        f"OpenCV incompleto o importado desde otra ruta: "
        f"{getattr(cv2, '__file__', None)}. "
        "Sigue la reparación de OpenCV indicada encima de esta celda."
    )

detector = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)
if detector.empty():
    raise RuntimeError("No se pudo cargar el detector facial.")

def preprocess(path):
    with Image.open(path) as image:
        rgb = np.asarray(ImageOps.exif_transpose(image).convert("RGB"))
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    faces = detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5,
                                     minSize=(40, 40))
    if len(faces) != 1:
        return None
    x, y, w, h = faces[0]
    face = cv2.resize(rgb[y:y+h, x:x+w], (SIZE, SIZE),
                      interpolation=cv2.INTER_AREA)
    return face.astype(np.float32) / 255.0

def load_dataset(root):
    groups = {}
    seen_ids, seen_files = set(), set()
    minimums = {"train": (2, 2), "cal": (2, 4), "test": (2, 4),
                "unknown": (1, 1)}
    for split, (min_ids, min_photos) in minimums.items():
        data = defaultdict(list)
        rejected = 0
        for folder in sorted((root / split).glob("*/")):
            if not folder.is_dir() or folder.name.startswith("."):
                continue
            identity = folder.name.casefold()
            if identity in seen_ids:
                raise ValueError(f"Identidad repetida entre particiones: {identity}")
            seen_ids.add(identity)
            for path in sorted(folder.iterdir()):
                if path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
                    continue
                digest = hashlib.sha256(path.read_bytes()).digest()
                if digest in seen_files:
                    raise ValueError(f"Imagen duplicada por contenido: {path}")
                seen_files.add(digest)
                face = preprocess(path)
                if face is None:
                    rejected += 1
                else:
                    data[identity].append(face)
            if len(data[identity]) < min_photos:
                raise ValueError(f"{split}/{identity} requiere {min_photos} fotos válidas.")
        if len(data) < min_ids:
            raise ValueError(f"{split} requiere al menos {min_ids} identidades.")
        print(f"{split}: {len(data)} identidades, "
              f"{sum(map(len, data.values()))} fotos válidas, {rejected} descartadas")
        groups[split] = data
    return groups


In [6]:
def sample_triplets(data, rng, count):
    keys = list(data)
    batch = [[], [], []]
    for _ in range(count):
        same, other = rng.choice(keys, 2, replace=False)
        a, p = rng.choice(len(data[same]), 2, replace=False)
        n = rng.integers(len(data[other]))
        for destination, image in zip(batch, (data[same][a], data[same][p], data[other][n])):
            destination.append(image)
    return tuple(np.stack(images) for images in batch)

def embed_groups(encoder, groups):
    return {identity: encoder.predict(np.stack(images), verbose=0)
            for identity, images in groups.items()}

def gallery_and_queries(groups, seed):
    rng = np.random.default_rng(seed)
    gallery, queries = {}, {}
    for identity, embeddings in groups.items():
        order = rng.permutation(len(embeddings))
        gallery[identity] = embeddings[order[:3]]
        queries[identity] = embeddings[order[3:]]
    return gallery, queries

def scores(gallery, queries, shots):
    ids = list(gallery)
    templates = np.stack([gallery[i][:shots].mean(axis=0) for i in ids])
    templates /= np.maximum(np.linalg.norm(templates, axis=1, keepdims=True), 1e-8)
    labels = np.concatenate([[i] * len(v) for i, v in queries.items()])
    query_vectors = np.concatenate(list(queries.values()))
    distances = ((query_vectors[:, None] - templates[None, :])**2).sum(axis=-1)
    return np.array(ids), labels, distances

def calibrate(ids, labels, distances):
    genuine = distances[labels[:, None] == ids]
    impostor = distances[labels[:, None] != ids]
    unique = np.unique(distances)
    candidates = np.r_[-1e-6, unique]
    errors = [(np.mean(impostor <= t) + np.mean(genuine > t)) / 2
              for t in candidates]
    return float(candidates[np.argmin(errors)])

def evaluate(encoder, groups):
    cal_g, cal_q = gallery_and_queries(embed_groups(encoder, groups["cal"]), 17)
    test_g, test_q = gallery_and_queries(embed_groups(encoder, groups["test"]), 23)
    unknown = embed_groups(encoder, groups["unknown"])
    print("Modalidad | Umbral | Top-1 | FAR pares | FRR pares | Rechazo desconocidos")
    for shots in (1, 3):
        threshold = calibrate(*scores(cal_g, cal_q, shots))
        ids, labels, distances = scores(test_g, test_q, shots)
        mask = labels[:, None] == ids
        top1 = np.mean(ids[distances.argmin(axis=1)] == labels)
        far = np.mean(distances[~mask] <= threshold)
        frr = np.mean(distances[mask] > threshold)
        _, _, unknown_distances = scores(test_g, unknown, shots)
        reject = np.mean(unknown_distances.min(axis=1) > threshold)
        print(f"{shots}-shot | {threshold:.4f} | {top1:.1%} ({len(labels)} consultas)"
              f" | {far:.1%} ({(~mask).sum()} pares) | {frr:.1%} ({mask.sum()} pares)"
              f" | {reject:.1%} ({len(unknown_distances)} consultas)")


## 5. Entrenamiento y evaluación

Cambia la ruta y la confirmación solo cuando tengas autorización para todas las fotos. Una vez entrenado, compara consultas con una galería de identidades nuevas para el codificador. One-shot usa una foto de referencia. Few-shot usa tres y normaliza su media. Calibración fija el umbral por separado para cada modalidad. Top-1 mide identificación cerrada antes del umbral. FAR y FRR corresponden a comparaciones por pares. El rechazo de desconocidos es una prueba de conjunto abierto, no una identificación zero-shot. Para identificar un rostro sin fotos de referencia haría falta información auxiliar y un modelo que la relacione con el espacio visual.

## Evidencias para el informe

Guarda las salidas de conteo del dataset, pérdida por época y tabla de métricas. Para las capturas, usa estas salidas numéricas con identificadores anónimos. El repositorio debe contener este notebook y puede incluir el PDF. El dataset se mantiene en el entorno privado.


In [40]:
def detectar_rostros(gris):
    return detector.detectMultiScale(
        gris, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40)
    )

def procesar_foto(ruta):
    with Image.open(ruta) as foto:
        rgb = np.array(ImageOps.exif_transpose(foto).convert("RGB"))

    gris = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    rostros = detectar_rostros(gris)

    if len(rostros) == 0:
        rgb = cv2.resize(rgb, None, fx=2, fy=2, interpolation=cv2.INTER_LINEAR)
        gris = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
        gris = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gris)
        rostros = detectar_rostros(gris)

    if len(rostros) != 1:
        return None, len(rostros)

    x, y, w, h = rostros[0]
    recorte = cv2.resize(rgb[y:y+h, x:x+w], (SIZE, SIZE))
    return recorte.astype(np.float32) / 255.0, 1

def preprocess(ruta):
    return procesar_foto(ruta)[0]

carpeta = Path("/content/dataset_privado/test/persona06")
validas = 0

for foto in sorted(carpeta.iterdir()):
    if foto.is_file() and foto.suffix.lower() in {".jpg", ".jpeg", ".png"}:
        imagen, cantidad = procesar_foto(foto)
        validas += int(imagen is not None)
        estado = "válida" if imagen is not None else "descartada"
        print(f"{foto.name}: {estado} ({cantidad} rostros detectados)")

print(f"Total: {validas} fotos válidas. Se requieren 4.")

image25.jpeg: válida (1 rostros detectados)
image26.jpeg: válida (1 rostros detectados)
image27.jpeg: descartada (2 rostros detectados)
image28.jpeg: descartada (2 rostros detectados)
image29.jpeg: descartada (0 rostros detectados)
image30.jpeg: válida (1 rostros detectados)
image31.jpeg: válida (1 rostros detectados)
image32.jpeg: válida (1 rostros detectados)
image33.jpeg: válida (1 rostros detectados)
Total: 6 fotos válidas. Se requieren 4.


In [41]:
DATASET = Path("/content/dataset_privado")
CONFIRMO_CONSENTIMIENTO = True
EPOCHS, STEPS, BATCH = 15, 20, 16

if CONFIRMO_CONSENTIMIENTO:
    keras.utils.set_random_seed(42)
    rng = np.random.default_rng(42)
    groups = load_dataset(DATASET)
    encoder = build_encoder()
    optimizer = keras.optimizers.Adam(learning_rate=0.0003)
    for epoch in range(1, EPOCHS + 1):
        losses = []
        for _ in range(STEPS):
            triplets = sample_triplets(groups["train"], rng, BATCH)
            losses.append(float(train_step(encoder, optimizer, *triplets).numpy()))
        print(f"Época {epoch:02d} | pérdida media {np.mean(losses):.4f}")
    evaluate(encoder, groups)
else:
    print("Evaluación facial pendiente: carga el dataset autorizado y confirma el consentimiento.")


train: 2 identidades, 4 fotos válidas, 0 descartadas
cal: 2 identidades, 8 fotos válidas, 0 descartadas
test: 2 identidades, 9 fotos válidas, 4 descartadas
unknown: 1 identidades, 1 fotos válidas, 0 descartadas
Época 01 | pérdida media 0.0118
Época 02 | pérdida media 0.0000
Época 03 | pérdida media 0.0000
Época 04 | pérdida media 0.0000
Época 05 | pérdida media 0.0000
Época 06 | pérdida media 0.0000
Época 07 | pérdida media 0.0000
Época 08 | pérdida media 0.0000
Época 09 | pérdida media 0.0000
Época 10 | pérdida media 0.0000
Época 11 | pérdida media 0.0000
Época 12 | pérdida media 0.0000
Época 13 | pérdida media 0.0000
Época 14 | pérdida media 0.0000
Época 15 | pérdida media 0.0000
Modalidad | Umbral | Top-1 | FAR pares | FRR pares | Rechazo desconocidos
1-shot | 0.0772 | 66.7% (3 consultas) | 100.0% (3 pares) | 0.0% (3 pares) | 0.0% (1 consultas)
3-shot | -0.0000 | 33.3% (3 consultas) | 0.0% (3 pares) | 100.0% (3 pares) | 100.0% (1 consultas)
